# Heading fusion experiments
Two alternative ROS heading filters: scalar heading and heading plus gyro bias. Run one variant in a fresh kernel; do not run all cells as a single pipeline. Inputs use the project-specific Pose2D heading in degrees. The live pose node is not replaced automatically.

In [ ]:
import math
import time

import rclpy
from rclpy.node import Node
from geometry_msgs.msg import Pose2D
from sensor_msgs.msg import Imu


In [ ]:

# ----------------------------
# Helpers (no def blocks; using lambdas)
# ----------------------------
wrap_to_pi = lambda a: (a + math.pi) % (2*math.pi) - math.pi
ang_diff   = lambda a, b: wrap_to_pi(a - b)   # smallest signed difference a-b

deg2rad = math.pi / 180.0
rad2deg = 180.0 / math.pi

# ----------------------------
# ROS2 init
# ----------------------------
rclpy.init()
node = Node("heading_kf_simple")

# ----------------------------
# Latest sensor data holders
# ----------------------------
enc_theta_deg = None
imu_theta_deg = None
gyro_z = None  # rad/s

enc_new = False
imu_new = False
gyro_new = False

# minimal "callbacks" via lambda (still required by rclpy)
sub_enc = node.create_subscription(
    Pose2D, "/robot/pose_enc",
    lambda msg: globals().update(enc_theta_deg=msg.theta, enc_new=True),
    10
)

sub_imu_heading = node.create_subscription(
    Pose2D, "/robot/pose",
    lambda msg: globals().update(imu_theta_deg=msg.theta, imu_new=True),
    10
)

sub_imu_raw = node.create_subscription(
    Imu, "/robot/imu_raw",
    lambda msg: globals().update(gyro_z=msg.angular_velocity.z, gyro_new=True),
    50
)

# (optional) publish fused heading
pub_fused = node.create_publisher(Pose2D, "/robot/pose_fused", 10)

# ----------------------------
# KF parameters (TUNE THESE)
# ----------------------------
# State: theta (radians)
theta_hat = 0.0          # X_k
P = 1.0                  # C_k (uncertainty). Start "kinda uncertain"

# Process noise: how much heading uncertainty grows each step
# If gyro is noisy / lots of unmodeled stuff, increase Q
Q = (0.5 * deg2rad)**2   # rad^2 per step (rough starting point)

# Measurement noises (sensor trust)
R_enc = (2.0 * deg2rad)**2   # encoder heading noise
R_imu = (5.0 * deg2rad)**2   # IMU heading noise (if you use it)

# ----------------------------
# Initialize theta from first available heading
# ----------------------------
print("Waiting for first heading message...")
while rclpy.ok() and enc_theta_deg is None and imu_theta_deg is None:
    rclpy.spin_once(node, timeout_sec=0.1)

if enc_theta_deg is not None:
    theta_hat = wrap_to_pi(enc_theta_deg * deg2rad)
else:
    theta_hat = wrap_to_pi(imu_theta_deg * deg2rad)

t_prev = time.time()

print("KF running. Publishing /robot/pose_fused (Pose2D). Ctrl+C to stop.")

# ----------------------------
# Main loop
# ----------------------------
try:
    while rclpy.ok():
        rclpy.spin_once(node, timeout_sec=0.01)

        # compute dt
        t_now = time.time()
        dt = t_now - t_prev
        if dt <= 0.0:
            dt = 1e-3
        t_prev = t_now

        # ============================================================
        # 1) STATE PREDICTION: X_{k+1|k} = F_k X_k + G_k U_k
        # For theta: theta = theta + gyro_z * dt
        # Here: X_k = theta_hat
        #       F_k = 1
        #       U_k = gyro_z
        #       G_k = dt
        # ============================================================
        if gyro_z is not None:
            theta_hat = wrap_to_pi(theta_hat + gyro_z * dt)

        # ============================================================
        # 2) COVARIANCE PREDICTION: C_{k+1|k} = F C_k F^T + Q
        # Here F=1, so: P = P + Q
        # (You can scale Q with dt if you want: P += Q*dt, but keep it simple first)
        # ============================================================
        P = P + Q

        # ============================================================
        # Measurement update helper (inline, repeated twice)
        #   Z_p = H X_{k+1|k}   (predicted measurement)
        #   Inno = Z - Z_p      (innovation / residual)  <-- we use this convention
        #   S = H P H^T + R
        #   K = P H^T S^-1
        #   X = X + K Inno
        #   P = (I - K H) P
        #
        # For heading measurement: H = 1
        # ============================================================

        # ---------- Update with encoder heading ----------
        if enc_new and enc_theta_deg is not None:
            enc_new = False

            z = wrap_to_pi(enc_theta_deg * deg2rad)  # measurement Z
            z_p = theta_hat                          # Z_p = H*X, with H=1
            inno = ang_diff(z, z_p)                  # innovation (wrapped)

            S = P + R_enc
            K = P / S

            theta_hat = wrap_to_pi(theta_hat + K * inno)
            P = (1.0 - K) * P

        # ---------- Update with IMU heading (optional) ----------
        # If your /robot/pose heading is from integrating the SAME gyro,
        # it's correlated with the prediction. This can still "work" but can be overconfident.
        # You can comment this block out if you only want encoder correction.
        if imu_new and imu_theta_deg is not None:
            imu_new = False

            z = wrap_to_pi(imu_theta_deg * deg2rad)
            z_p = theta_hat
            inno = ang_diff(z, z_p)

            S = P + R_imu
            K = P / S

            theta_hat = wrap_to_pi(theta_hat + K * inno)
            P = (1.0 - K) * P

        # publish fused
        msg_out = Pose2D()
        msg_out.x = 0.0
        msg_out.y = 0.0
        msg_out.theta = (theta_hat * rad2deg) % 360.0  # back to 0..360 deg
        pub_fused.publish(msg_out)

        # slow print (optional)
        # print(f"theta_fused_deg={msg_out.theta:.2f}  P={P:.6f}")

        time.sleep(0.005)

except KeyboardInterrupt:
    pass

node.destroy_node()
rclpy.shutdown()
print("Stopped.")


In [ ]:
import time
import numpy as np

import rclpy
from geometry_msgs.msg import Pose2D
from sensor_msgs.msg import Imu


In [ ]:


# -----------------------------
# Angle helpers
# -----------------------------
def wrap_to_pi(a):
    """Wrap angle (rad) to (-pi, pi]."""
    return (a + np.pi) % (2*np.pi) - np.pi

def wrap_to_360_deg(d):
    """Wrap angle (deg) to [0, 360)."""
    return d % 360.0

def angle_diff_deg(a_deg, b_deg):
    """Smallest difference a-b in degrees, result in (-180, 180]."""
    d = (a_deg - b_deg + 180.0) % 360.0 - 180.0
    return d

def angle_diff_rad(a_rad, b_rad):
    """Smallest difference a-b in radians, result in (-pi, pi]."""
    return wrap_to_pi(a_rad - b_rad)


# -----------------------------
# Tuning parameters (START HERE)
# -----------------------------
# Process noise (model uncertainty)
q_theta = 0.02   # (rad^2 / s)   increase if KF feels too "stiff"/slow to follow turns
q_bias  = 0.001  # (rad^2 / s^3) increase if bias changes over time

# Measurement noises (sensor trust)
R_enc = (np.deg2rad(2.0))**2    # encoder heading variance (rad^2)
R_imu = (np.deg2rad(5.0))**2    # imu heading variance (rad^2) (often noisier/drifty)

USE_IMU_HEADING_UPDATE = True   # set False if you only want to correct with encoder


# -----------------------------
# Kalman Filter state
# x = [theta, bias]^T
# -----------------------------
x = np.array([0.0, 0.0], dtype=float)      # [theta(rad), bias(rad/s)]
P = np.diag([ (np.deg2rad(30.0))**2, (0.5)**2 ])  # big initial uncertainty

I2 = np.eye(2)

# Latest sensor values
imu_heading_deg = None
enc_heading_deg = None
gyro_z = None

new_imu_heading = False
new_enc_heading = False
new_gyro = False

last_time = None
last_print = 0.0


# -----------------------------
# ROS callbacks
# -----------------------------
def cb_pose_imu(msg: Pose2D):
    global imu_heading_deg, new_imu_heading
    imu_heading_deg = wrap_to_360_deg(msg.theta)  # Pose2D.theta is your heading in deg (per your setup)
    new_imu_heading = True

def cb_pose_enc(msg: Pose2D):
    global enc_heading_deg, new_enc_heading
    enc_heading_deg = wrap_to_360_deg(msg.theta)
    new_enc_heading = True

def cb_imu_raw(msg: Imu):
    global gyro_z, new_gyro
    gyro_z = msg.angular_velocity.z  # rad/s
    new_gyro = True


# -----------------------------
# One KF predict step
# -----------------------------
def kf_predict(dt, gyro_z_meas):
    """
    Predict:
      theta_{k+1} = theta_k + (gyro_z - bias_k)*dt
      bias_{k+1}  = bias_k
    """
    global x, P

    # State transition F and input B for u = gyro_z
    F = np.array([[1.0, -dt],
                  [0.0,  1.0]], dtype=float)

    B = np.array([dt, 0.0], dtype=float)  # multiplies gyro_z scalar

    # Process noise Q (simple model)
    Q = np.array([[q_theta * dt,      0.0],
                  [0.0,          q_bias * dt]], dtype=float)

    # Predict state
    x = F @ x + B * gyro_z_meas
    x[0] = wrap_to_pi(x[0])

    # Predict covariance
    P = F @ P @ F.T + Q


# -----------------------------
# One KF update with heading measurement
# -----------------------------
def kf_update_heading(z_heading_deg, R):
    """
    Measurement: z = theta + noise
    H = [1, 0]
    """
    global x, P

    # Convert measurement to rad and wrap
    z = wrap_to_pi(np.deg2rad(z_heading_deg))

    # Innovation (angle-aware)
    y = angle_diff_rad(z, x[0])  # z - theta_pred wrapped

    # H is 1x2
    H = np.array([1.0, 0.0], dtype=float)

    # Innovation covariance S (scalar)
    S = H @ P @ H.T + R

    # Kalman gain K (2x1)
    K = (P @ H.T) / S  # shape (2,)

    # Update state
    x = x + K * y
    x[0] = wrap_to_pi(x[0])

    # Update covariance
    P = (I2 - np.outer(K, H)) @ P


# -----------------------------
# Main loop (spin + KF)
# -----------------------------
rclpy.init()
node = rclpy.create_node("heading_kf_notebook")

sub1 = node.create_subscription(Pose2D, "/robot/pose", cb_pose_imu, 10)
sub2 = node.create_subscription(Pose2D, "/robot/pose_enc", cb_pose_enc, 10)
sub3 = node.create_subscription(Imu, "/robot/imu_raw", cb_imu_raw, 50)

print("Running... Press Stop/Interrupt in Jupyter to end.\n")

try:
    while rclpy.ok():
        rclpy.spin_once(node, timeout_sec=0.01)

        now = time.time()
        if last_time is None:
            last_time = now
            continue

        dt = now - last_time
        if dt <= 0.0:
            continue
        last_time = now

        # 1) PREDICT using gyro_z if available
        if gyro_z is not None:
            kf_predict(dt, gyro_z)

        # 2) UPDATE with encoder heading when a new one arrives
        if new_enc_heading and enc_heading_deg is not None:
            kf_update_heading(enc_heading_deg, R_enc)
            new_enc_heading = False

        # 3) OPTIONAL UPDATE with IMU heading when a new one arrives
        if USE_IMU_HEADING_UPDATE and new_imu_heading and imu_heading_deg is not None:
            kf_update_heading(imu_heading_deg, R_imu)
            new_imu_heading = False

        # 4) Print at ~10 Hz
        if now - last_print > 0.1:
            last_print = now

            kf_heading_deg = wrap_to_360_deg(np.degrees(x[0]))

            imu_str = f"{imu_heading_deg:7.2f}" if imu_heading_deg is not None else "   None "
            enc_str = f"{enc_heading_deg:7.2f}" if enc_heading_deg is not None else "   None "

            print(f"IMU heading: {imu_str} deg | ENC heading: {enc_str} deg | KF heading: {kf_heading_deg:7.2f} deg")

except KeyboardInterrupt:
    print("\nStopped by user.")

finally:
    node.destroy_node()
    rclpy.shutdown()
